# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a demonstration of loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing Croissant schema `@id` fields throughout.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs, referencing all entities by their `@id` fields.

- `record_sets`: Collection of record sets in the dataset (each has an `@id`)
- `fields`: Collection of fields for each record set (each has an `@id`)
- `columns`: Each field refers to a column with an `@id`

In [ ]:
# List all record sets and their field @ids
record_set_objects = list(dataset.record_sets)

for rs in record_set_objects:
    print(f"RecordSet @id: {rs['@id']}  | Name: {rs.get('name', '(no name)')}")
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            if isinstance(f, dict):
                print(f"    Field @id: {f['@id']}  | Name: {f.get('name', '(no name)')}")
                if 'column' in f:
                    cols = f['column']
                    if isinstance(cols, list):
                        for c in cols:
                            if isinstance(c, dict):
                                print(f"      Column @id: {c['@id']}  | Name: {c.get('name', '(no name)')}")
                            else:
                                print(f"      Column @id: {c}")
                    else:
                        if isinstance(cols, dict):
                            print(f"      Column @id: {cols['@id']}  | Name: {cols.get('name', '(no name)')}")
                        else:
                            print(f"      Column @id: {cols}")
            else:
                print(f"    Field @id: {f}")
    else:
        print("   (no fields)")
    print("-")

### Get and Preview Example Record Set
Let us enumerate the record set `@id`s for use in downstream extraction.

In [ ]:
# List all record set @ids for extraction
record_set_ids = [rs['@id'] for rs in record_set_objects]
print("Available Record Set @ids:")
for idx, rset_id in enumerate(record_set_ids):
    print(f"  [{idx}] {rset_id}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use each record set's `@id` retrieved above.

In [ ]:
# Extract data from all record sets and preview example counts and columns
dataframes = {}
for rid in record_set_ids:
    df = pd.DataFrame(dataset.records(record_set=rid))
    dataframes[rid] = df
    print(f"Loaded RecordSet @id: {rid}; #rows: {len(df)}; Columns: {list(df.columns)}")
    if len(df) > 0:
        display(df.head())
    print("---")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering and transforming data using column `@id`s. For illustration, we attempt common EDA tasks such as numeric normalization and groupby, referencing fields by their `@id`s.

> **Note:** If unsure which record set contains quantitative fields, re-inspect column names and types above. Adjust the code below to point to the appropriate record set and column `@id` as shown.

In [ ]:
# Example: Use the first record set containing a numeric field for EDA
# Update these to point to an actual numeric field/column @id from your dataset

# Choose one record set with numeric columns
selected_rs = None
numeric_col = None
group_col = None
for rs_id, df in dataframes.items():
    for col in df.columns:
        # Try to select an integer or float column
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                selected_rs = rs_id
                numeric_col = col
                # For grouping, use another distinct column if possible
                for group_candidate in df.columns:
                    if group_candidate != col and df[group_candidate].nunique() > 1 and df[group_candidate].nunique() < 10:
                        group_col = group_candidate
                        break
                break
        except Exception:
            continue
    if selected_rs and numeric_col:
        break

if selected_rs is None:
    print("No numeric field found for EDA.")
else:
    print(f"Using RecordSet @id: {selected_rs}")
    print(f"Numeric field @id: {numeric_col}")
    if group_col:
        print(f"Group field @id: {group_col}")
    df = dataframes[selected_rs].copy()
    threshold = df[numeric_col].mean()  # use mean as example threshold
    filtered_df = df[df[numeric_col] > threshold]
    print(f"Filtered records with {numeric_col} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
    print(f"Normalized {numeric_col} for filtered records:")
    display(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

    if group_col and group_col in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_col)[numeric_col].mean().reset_index().rename(columns={numeric_col: f"mean_{numeric_col}"})
        print(f"Grouped data by {group_col} (mean of {numeric_col}):")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships using referenced `@id` columns. Below are some example plots (adjust `numeric_col` and `group_col` if needed).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if selected_rs and numeric_col in dataframes[selected_rs].columns:
    df = dataframes[selected_rs]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_col], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_col} (@id)")
    plt.xlabel(numeric_col)
    plt.ylabel("Count")
    plt.show()

    if group_col and group_col in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_col], y=df[numeric_col])
        plt.title(f"{numeric_col} by {group_col} (@id)")
        plt.xlabel(group_col)
        plt.ylabel(numeric_col)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and explore the FAIR² colorectal cancer dataset package, referencing all entities by their `@id`. We performed a survey of available record sets and fields, extracted records by ID, and carried out basic exploratory data analysis and visualization. Further biomedical or statistical modeling can proceed, leveraging the schema's consistent `@id` referencing throughout your workflow.